# Лабораторна робота №2. Петренко Денис - ФБ-з41
## Збір даних рослин України за областями.

In [2]:
# Підготовка середовища
%pip install notebook pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 53.6 MB/s eta 0:00:00


### Підготовка середовища
Імпорт необхідних бібліотек та створення словників для роботи з областями

In [6]:
import urllib.request
import os
import hashlib
import glob
from datetime import datetime
import pandas as pd

DATA_DIR = "vhi_data"
SKIP_DOWNLOAD = False

REGIONS = {
    1: "Cherkasy", 2: "Chernihiv", 3: "Chernivtsi", 4: "Crimea", 5: "Dnipropetrovsk",
    6: "Donetsk", 7: "Ivano-Frankivsk", 8: "Kharkiv", 9: "Kherson", 10: "Khmelnytskyy",
    11: "Kyiv", 12: "Kyiv_City", 13: "Kirovohrad", 14: "Luhansk", 15: "Lviv",
    16: "Mykolayiv", 17: "Odesa", 18: "Poltava", 19: "Rivne", 20: "Sevastopol",
    21: "Sumy", 22: "Ternopil", 23: "Transcarpathia", 24: "Vinnytsya", 25: "Volyn",
    26: "Zaporizhzhya", 27: "Zhytomyr"
}

ukr_mapping = {
    "Vinnytsya": 1, "Volyn": 2, "Dnipropetrovsk": 3, "Donetsk": 4,
    "Zhytomyr": 5, "Transcarpathia": 6, "Zaporizhzhya": 7,
    "Ivano-Frankivsk": 8, "Kyiv": 9, "Kirovohrad": 10,
    "Luhansk": 11, "Lviv": 12, "Mykolayiv": 13, "Odesa": 14,
    "Poltava": 15, "Rivne": 16, "Sumy": 17, "Ternopil": 18,
    "Kharkiv": 19, "Kherson": 20, "Khmelnytskyy": 21,
    "Cherkasy": 22, "Chernivtsi": 23, "Chernihiv": 24,
    "Crimea": 25, "Kyiv_City": 26, "Sevastopol": 27
}

### Завдання 1: Завантаження даних
Для кожної з адміністративних одиниць України завантажено структуровані файли з індексом VHI.
Реалізовано механізм запобігання повторного завантаження (перевірка хешу) та колізії даних. Індекс 0 пропускається.

In [8]:
def download_vhi_data(skip_download=False):
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)

    for prov_id, prov_name in REGIONS.items():
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={prov_id}&year1=1981&year2=2024&type=Mean"

        try:
            with urllib.request.urlopen(url) as response:
                new_data = response.read()

            new_hash = hashlib.md5(new_data).hexdigest()
            old_files = glob.glob(os.path.join(DATA_DIR, f"vhi_id_{prov_id}_*.csv"))
            needs_update = True

            if old_files:
                with open(old_files[-1], "rb") as f:
                    old_hash = hashlib.md5(f.read()).hexdigest()
                if new_hash == old_hash:
                    needs_update = False

            if needs_update:

                for old_file in old_files:
                    os.remove(old_file)

                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = os.path.join(DATA_DIR, f"vhi_id_{prov_id}_{timestamp}.csv")
                print(f' + Оновлення файлу {os.path.join(DATA_DIR, f"vhi_id_{prov_id}_{timestamp}.csv")}...')

                with open(filename + ".tmp", "wb") as f:
                    f.write(new_data)
                os.rename(filename + ".tmp", filename)
            else:
                print(f' - Файл {os.path.join(DATA_DIR, f"vhi_id_{prov_id}_*.csv")} актуальний, оновлення не потрібне!')

        except Exception as e:
            print(f"Error downloading {prov_name}: {e}")

# Виклик функції (skip_download = True, якщо треба використати файли, що були завантажені раніше)
download_vhi_data()


 + Оновлення файлу vhi_data/vhi_id_1_20260504_170357.csv...
 + Оновлення файлу vhi_data/vhi_id_2_20260504_170357.csv...
 + Оновлення файлу vhi_data/vhi_id_3_20260504_170359.csv...
 + Оновлення файлу vhi_data/vhi_id_4_20260504_170359.csv...
 + Оновлення файлу vhi_data/vhi_id_5_20260504_170359.csv...
 + Оновлення файлу vhi_data/vhi_id_6_20260504_170400.csv...
 + Оновлення файлу vhi_data/vhi_id_7_20260504_170400.csv...
 + Оновлення файлу vhi_data/vhi_id_8_20260504_170400.csv...
 + Оновлення файлу vhi_data/vhi_id_9_20260504_170400.csv...
 + Оновлення файлу vhi_data/vhi_id_10_20260504_170400.csv...
 + Оновлення файлу vhi_data/vhi_id_11_20260504_170401.csv...
 + Оновлення файлу vhi_data/vhi_id_12_20260504_170401.csv...
 + Оновлення файлу vhi_data/vhi_id_13_20260504_170401.csv...
 + Оновлення файлу vhi_data/vhi_id_14_20260504_170401.csv...
 + Оновлення файлу vhi_data/vhi_id_15_20260504_170402.csv...
 + Оновлення файлу vhi_data/vhi_id_16_20260504_170402.csv...
 + Оновлення файлу vhi_data/vhi_i

### Завдання 2: Очищення даних та заміна індексів
Зчитано завантажені файли у DataFrame. Прибрано зайві стовпці, заповнено пропуски, видалено HTML-теги.
Додано стовпчики з назвою та змінено індекси областей за українською абеткою.

In [10]:
def process_vhi_data():
    all_dfs = []
    files = glob.glob(os.path.join(DATA_DIR, "*.csv"))

    for file in files:
        prov_id = int(os.path.basename(file).split('_')[2])
        prov_name = REGIONS[prov_id]

        df = pd.read_csv(
            file,
            header=1,
            usecols=[0, 1, 2, 3, 4, 5, 6],
            names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
        )

        df['Year'] = df['Year'].astype(str).str.replace(r'<[^>]*>', '', regex=True)
        df['VHI'] = df['VHI'].astype(str).str.replace(r'<[^>]*>', '', regex=True)

        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
        df['VHI'] = pd.to_numeric(df['VHI'], errors='coerce')

        df = df.dropna(subset=['Year', 'VHI'])

        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)

        df['Region_Name'] = prov_name
        df['Region_ID'] = ukr_mapping[prov_name]

        df = df[['Region_ID', 'Region_Name', 'Year', 'Week', 'VCI', 'TCI', 'VHI']]
        all_dfs.append(df)

    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    return pd.DataFrame()

df_clean = process_vhi_data()
df_clean = df_clean.sort_values(by=['Region_ID', 'Year', 'Week']).reset_index(drop=True)
print(f"Дані успішно оброблено! Всього записів: {len(df_clean)}")
df_clean.head()

Дані успішно оброблено! Всього записів: 60372


,Region_ID,Region_Name,Year,Week,VCI,TCI,VHI
0,1,Vinnytsya,1982,1,63.47,28.34,45.90
1,1,Vinnytsya,1982,2,67.62,23.05,45.34
2,1,Vinnytsya,1982,3,69.37,20.40,44.88
3,1,Vinnytsya,1982,4,65.26,17.93,41.60
4,1,Vinnytsya,1982,5,58.58,20.00,39.29


### Завдання 3.1: Вибірка ряду VHI для області за вказаний рік
Процедура повертає тижні та значення VHI для однієї обраної області та року.

In [11]:
def get_vhi_series_by_year(df, region_id, year):
    return df[(df['Region_ID'] == region_id) & (df['Year'] == year)][['Week', 'VHI']]

print("VHI для Кіровоградської області (2021 рік):")
get_vhi_series_by_year(df_clean, 10, 2021).head(10)

VHI для Кіровоградської області (2021 рік):


,Week,VHI
22152,1,48.13
22153,2,50.85
22154,3,53.01
22155,4,54.44
22156,5,53.34
22157,6,52.72
22158,7,53.20
22159,8,52.27
22160,9,50.75
22161,10,49.36


### Завдання 3.2: Вибірка VHI за вказаний діапазон років для вказаних областей
Процедура фільтрує загальний DataFrame за списком ID областей та проміжком часу.

In [12]:
def get_vhi_for_regions_and_years(df, region_ids, start_year, end_year):
    condition = (df['Region_ID'].isin(region_ids)) & (df['Year'] >= start_year) & (df['Year'] <= end_year)
    return df[condition][['Region_ID', 'Region_Name', 'Year', 'Week', 'VHI']].reset_index(drop=True)

# Тест: Вінницька (1) та Волинська (2) області за період 2010-2015 рр.
print("VHI для Вінницької та Волинської областей (2010-2015):")
get_vhi_for_regions_and_years(df_clean, [1, 2], 2010, 2015)

VHI для Вінницької та Волинської областей (2010-2015):


,Region_ID,Region_Name,Year,Week,VHI
0,1,Vinnytsya,2010,1,52.04
1,1,Vinnytsya,2010,2,51.05
2,1,Vinnytsya,2010,3,52.37
3,1,Vinnytsya,2010,4,52.69
4,1,Vinnytsya,2010,5,53.16
...,...,...,...,...,...
619,2,Volyn,2015,48,46.32
620,2,Volyn,2015,49,47.76
621,2,Volyn,2015,50,48.58
622,2,Volyn,2015,51,50.42


### Завдання 3.3: Пошук екстремумів, середнього та медіани
Розрахунок базової статистики VHI для вказаних областей та років.

In [13]:
def get_vhi_statistics(df, region_ids, start_year, end_year):
    filtered_df = df[(df['Region_ID'].isin(region_ids)) & (df['Year'] >= start_year) & (df['Year'] <= end_year)]
    if filtered_df.empty:
        return "Немає даних за вказаними критеріями."

    # Розраховуємо статистику та перетворюємо у DataFrame
    stats = filtered_df['VHI'].agg(['min', 'max', 'mean', 'median']).to_frame().T
    stats.index = ['VHI_Statistics']
    return stats

# Статистика для Київської області (ID = 9) за 2019-2024 роки
print("Статистика VHI для Київської області (2019-2024):")
get_vhi_statistics(df_clean, [9], 2019, 2024)

Статистика VHI для Київської області (2019-2024):


,min,max,mean,median
VHI_Statistics,20.91,65.72,45.126955,44.3
